# Notebook 02 — Baselines and throughput budget

**Purpose.** Metric sanity checks, B0/B1 CPU baselines, B2 (compact, uniform, fixed fusion) and B3 (MobileNetV3-Small) development fits on train with tune selection, throughput and the six-final-fit cost forecast. **Partitions:** train, tune only. **GPU:** bounded development pool (ledger stage `baselines`). Test labels are inaccessible to this role (`PartitionGuard`).

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


## Metric sanity checks (synthetic)

In [ ]:
import numpy as np
from cape_eeg import metrics as M
q = np.array([[1,0,0,0,0,0],[0.5,0.5,0,0,0,0]]); print('exact agreement KL', M.kl_rows(q, q)); print('uniform prediction KL', M.kl_rows(q, np.full((2,6), 1/6)), '= log6 - H(q) =', np.log(6) - np.array([0, np.log(2)]))

## B0 train prior and B1 band-power soft regression (CPU)

In [ ]:
run(['run_baselines_cpu.py'])

## Pretrained comparator provenance (timm / Hugging Face)

In [ ]:
import timm, torch
from cape_eeg.baselines import MobileNetV3Comparator
from cape_eeg.model import count_parameters
m = MobileNetV3Comparator(pretrained=True); print('timm', timm.__version__, 'model', MobileNetV3Comparator.HF_ID, 'adapted parameters', count_parameters(m))
print('adaptation: in_chans=4 (first conv re-initialised from the RGB kernel by timm), num_classes=6; no ImageNet colour normalisation is applied to EEG log-power inputs')

## B2 and B3 development fits (12 epochs, tune selection)

In [ ]:
for cfg in ['B2', 'B3']:
    run(['train.py', '--config', cfg, '--phase', 'dev', '--seed', '101', '--stage', 'baselines'])
    rd = sorted(ws.runs.glob(f'dev_{cfg}_s101_*'), key=lambda p: p.stat().st_mtime)[-1]
    run(['predict.py', '--run', rd.name, '--partitions', 'tune', '--views', 'full'])

## Throughput and final-fit cost forecast

In [ ]:
import pandas as pd
rows = []
for rd in sorted(ws.runs.glob('dev_*')):
    st = read_json(rd / 'status.json'); cfg = read_json(rd / 'config.resolved.json')
    if st and cfg: rows.append({'config': cfg['config_id'], 'status': st['status'], 'wall_min': round(st.get('wall_seconds', 0) / 60, 1), 'peak_cuda_gib': round(st.get('peak', {}).get('cuda_alloc_gib', 0), 2), 'peak_rss_gib': round(st.get('peak', {}).get('rss_gib', 0), 1), 'examples_seen': st.get('examples_seen'), 'best_epoch': (st.get('best') or {}).get('epoch'), 'tune_pKL': (st.get('best') or {}).get('tune_patient_kl')})
t = pd.DataFrame(rows); print(t.to_string(index=False))
from cape_eeg.training.supervisor import GpuLedger
led = GpuLedger(ws.runs / 'gpu_ledger.jsonl'); used = led.total_hours(); per_fit = t.wall_min.max() * 1.2 / 60 if len(t) else float('nan')
print(f'GPU hours used {used:.2f} / 12; projected six final fits (train+tune, +20%): {6 * per_fit:.2f} h; remaining after finals: {12 - used - 6 * per_fit:.2f} h')